## Phase 0 - Imports & Configuration

In [ ]:
import os
import json
import re
import string as _string
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import stanza
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

import fasttext
import fasttext.util
import compress_fasttext
import contextlib
import io
from gensim.models.fasttext import load_facebook_model

from captum.attr import IntegratedGradients
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
    precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
DATASET_PATH = "../PRDECT-ID Dataset.csv"
_SCRIPT_DIR  = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR    = os.path.join(_SCRIPT_DIR, "model_weights")
EVAL_DIR     = os.path.join(_SCRIPT_DIR, "evaluation_metrics")
RESULTS_DIR  = os.path.join(_SCRIPT_DIR, "results")
LSTM_MODEL_PATH = os.path.join(MODEL_DIR, "lstm_model.pt")
FT_MODEL_PATH   = os.path.join(MODEL_DIR, "fasttext_lstm_vectorizer.bin")

for d in (MODEL_DIR, EVAL_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

PAD_TOKEN = "[PAD]"
UNK_TOKEN = "[UNK]"
LABEL2ID  = {"negatif": 0, "positif": 1}
ID2LABEL  = {0: "negatif", 1: "positif"}

MAX_LENGTH    = 128
EMBEDDING_DIM = 300
HIDDEN_DIM    = 256
NUM_LAYERS    = 2
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS    = 5

CONFIDENCE_THRESHOLD = 0.50
SIMILARITY_THRESHOLD = 0.99
TOP_N_SUMMARY        = 10
MIN_COUNT            = 3

NEGATION_WORDS = {"tidak", "bukan", "belum", "jangan", "kurang", "tanpa"}
SPLIT_CONJUNCTIONS = ["tapi", "tetapi", "namun", "meski", "meskipun", "walaupun", "walau", "dan", "serta", "juga", "karena", "soalnya", "sebab", "padahal", "sedangkan"]

REGEX_PATTERNS = [
    (r"(.)\1{2,}",          r"\1\1"),   # "bangeeetttt" -> "bangeett"
    (r"([a-zA-Z])\1\b",     r"\1"),     # "bangett" -> "banget"
    (r"([!?,;.]){2,}",      r"\1"),     # ",,,,," -> ","
    (r"\b(\w+)\s+\1\b",     r"\1"),     # "bagus bagus" -> "bagus"
    (r"\s+([!?,;.])",       r"\1"),     # space before punctuation
    (r"([!?,;.])(?!\s)",    r"\1 "),    # space after punctuation
]

SLANG_DICT = {
    "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak", "engga": "tidak", "tdk": "tidak", "tak": "tidak",
    "tp": "tapi", "tpi": "tapi", "cuma": "hanya", "cmn": "hanya", "cuman": "hanya",
    "bgt": "banget", "bgd": "banget", "bngt": "banget", "bet": "banget", "bnget": "banget", "sgt": "sangat", "sngat": "sangat", "sangt": "sangat",
    "sy": "saya", "sya": "saya", "gw": "saya", "gue": "saya", "aku": "saya", "ak": "saya", "lo": "kamu", "lu": "kamu", "kmu": "kamu",
    "ud": "sudah", "udh": "sudah", "uda": "sudah", "udah": "sudah", "sdh": "sudah", "blm": "belum", "blum": "belum", "belom": "belum", "blom": "belum",
    "lg": "lagi", "lgi": "lagi", "bs": "bisa", "bsa": "bisa", "skrg": "sekarang", "skr": "sekarang", "skg": "sekarang", "cb": "coba", "cba": "coba",
    "br": "baru", "bru": "baru", "cpt": "cepat", "cepet": "cepat", "cpet": "cepat", "lm": "lama", "lma": "lama", "hrs": "harus", "hrus": "harus",
    "bgs": "bagus", "bgus": "bagus", "mntp": "bagus", "mntap": "bagus", "mantep": "bagus", "mntep": "bagus", "mantap": "bagus",
    "sj": "saja", "aja": "saja", "sja": "saja", "aj": "saja", "doank": "saja", "doang": "saja",
    "bbrp": "beberapa", "bbrapa": "beberapa", "bbrpa": "beberapa", "bebrpa": "beberapa", "brp": "berapa", "brpa": "berapa", "brapa": "berapa",
    "bgini": "seperti ini", "begini": "seperti ini", "sperti": "seperti", "yg": "yang", "yng": "yang", "krn": "karena", "karna": "karena",
    "dgn": "dengan", "dg": "dengan", "dr": "dari", "dri": "dari", "utk": "untuk", "buat": "untuk", "jg": "juga", "sm": "sama", "mk": "maka",
    "brg": "barang", "brng": "barang", "ongkir": "ongkos kirim", "ori": "original", "ok": "oke", "pngiriman": "pengiriman", "pngirim": "pengirim", "krm": "kirim",
    "saller": "penjual", "saler": "penjual", "seler": "penjual", "seller": "penjual", "pnjual": "penjual",
    "lmyn": "lumayan", "lmayan": "lumayan", "kualits": "kualitas", "packing": "packaging",
    "wkwk": "", "wkwkwk": "", "haha": "", "hihi": "", "hehe": "", "masyaallah": "", "sih": "", "alhamdulilah": "", "alhamdullilah": "", "alhamdulillah": "",
    "mantul": "bagus banget", "gokil": "luar biasa", "okesip": "oke siap"
}

STOPWORDS = {
    "yang", "di", "dan", "ini", "itu", "dengan", "untuk", "pada", "adalah", "dari", "dalam", "ke", "akan", "oleh", "saya", "aku", "baru",
    "kamu", "dia", "kami", "mereka", "nya", "ada", "sudah", "belum", "juga", "bisa", "hanya", "lebih", "lagi", "sangat", "sekali", "saat",
    "kalau", "jika", "atau", "karena", "tapi", "tetapi", "namun", "se", "si", "pun", "lah", "kah", "dong", "deh", "sih", "nih", "waktu",
    "ya", "yah", "kok", "kan", "mau", "mah", "banget", "aja", "udah", "jadi", "sama", "satu", "dua", "tiga", "masih", "harus", "banyak",
}

LSTM_LEXICON: set = set()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
stanza.download("id", verbose=False)
nlp_stanza = stanza.Pipeline(
    "id",
    processors="tokenize,mwt,pos,lemma,depparse",
    verbose=False,
)

## Phase 1 - Data Loading & Preprocessing

In [ ]:
_CLEAN_WORD_RE = re.compile(r'^[a-zA-Z]{3,}$')

def is_clean_word(word: str) -> bool:
    return bool(_CLEAN_WORD_RE.match(word))

def apply_regex_patterns(text: str) -> str:
    for pattern, replacement in REGEX_PATTERNS:
        text = re.sub(pattern, replacement, text)
    return text

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    words = text.split()
    normalized = []
    for word in words:
        replacement = SLANG_DICT.get(word, word)
        if replacement:
            normalized.extend(replacement.split())
    text = " ".join(normalized)
    text = apply_regex_patterns(text)
    text = re.sub(r"[^\w\s,.!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_and_clean_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df.rename(columns={"Customer Review": "review_text", "Customer Rating": "rating"})
    df["review_text"] = df["review_text"].astype(str)
    df["rating"]      = pd.to_numeric(df["rating"], errors="coerce")
    df = df.dropna(subset=["rating"])
    df["rating"] = df["rating"].astype(int)
    df = df[df["review_text"].str.split().str.len() >= 5]
    df = df[df["review_text"].str.contains(r"[a-zA-Z]", regex=True)]
    df = df[df["review_text"].str.contains(r"[a-zA-Z0-9\s.,!?]", regex=True)]
    df = df.drop_duplicates(subset=["review_text"])
    df = df.reset_index(drop=True)
    df["review_normalized"] = df["review_text"].apply(normalize_text)

    def assign_label(sentiment: str):
        if sentiment == "Negative": return 0
        elif sentiment == "Positive": return 1
        return None

    df["label"] = df["Sentiment"].apply(assign_label)

    print(f"Total after cleaning : {len(df)} rows")
    print(f"Label distribution:")
    print(f"  Positive (1): {(df['label'] == 1).sum()}")
    print(f"  Negative (0): {(df['label'] == 0).sum()}")
    return df

df_raw        = load_and_clean_data(DATASET_PATH)
df_train_pool = df_raw[df_raw["label"].notna()].copy()
df_train_pool["label"] = df_train_pool["label"].astype(int)

print()
print("Normalization samples:")
for i in range(min(3, len(df_raw))):
    print(f"ORIGINAL   : {df_raw['review_text'].iloc[i]}")
    print(f"NORMALIZED : {df_raw['review_normalized'].iloc[i]}")
    print()

## Phase 2 - LSTM Model Training

In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    df_train_pool['review_normalized'].values,
    df_train_pool['label'].values,
    test_size=0.10,
    random_state=42,
    stratify=df_train_pool['label'].values,
)
print(f"Stratified Splitting Complete (90:10):")
print(f"  Total Training samples: {len(X_train_raw)}")
print(f"  Total Testing samples : {len(X_test_raw)} (Locked)")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw.tolist(),
    y_train_raw.tolist(),
    test_size=0.10,
    random_state=42,
    stratify=y_train_raw,
)
print(f"  Train: {len(X_train)} | Val: {len(X_val)}")

def load_fasttext_model(ft_path: str):
    if os.path.exists(ft_path):
        print(f"Loading compressed FastText from: {ft_path}")
        ft = compress_fasttext.models.CompressedFastTextKeyedVectors.load(ft_path)
        return ft
    print("Downloading official Indonesian FastText model (cc.id.300.bin)...")
    with contextlib.redirect_stdout(io.StringIO()):
        fasttext.util.download_model("id", if_exists="ignore")
    full_model_path = "cc.id.300.bin"
    print("Loading model via Gensim...")
    full_ft_gensim = load_facebook_model(full_model_path).wv
    print("Compressing FastText model with compress-fasttext...")
    small_ft = compress_fasttext.prune_ft_freq(full_ft_gensim, pq=True, qdim=100)
    small_ft.save(ft_path)
    print(f"Compressed FastText saved to: {ft_path}")
    if os.path.exists(full_model_path):
        os.remove(full_model_path)
    return small_ft

ft_model = load_fasttext_model(FT_MODEL_PATH)

print("Building vocabulary from training tokens via Stanza...")

def tokenize_with_stanza(text: str) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp_stanza(text)
    return [w.lemma.lower() if w.lemma else w.text.lower() for sent in doc.sentences for w in sent.words]

vocab_dict = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for text in tqdm(X_train, desc="Stanza tokenisation for vocab"):
    for token in tokenize_with_stanza(text):
        if token not in vocab_dict:
            vocab_dict[token] = len(vocab_dict)

vocab_size = len(vocab_dict)
print(f"Vocabulary size: {vocab_size} tokens")

print("Building embedding matrix from FastText vectors...")
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

for word, idx in vocab_dict.items():
    if word in (PAD_TOKEN, UNK_TOKEN):
        if word == UNK_TOKEN:
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(EMBEDDING_DIM,))
        continue
    try:
        embedding_matrix[idx] = ft_model[word]
    except KeyError:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(EMBEDDING_DIM,))

print(f"Embedding matrix shape: {embedding_matrix.shape}")

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, pretrained_embeddings=None, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained_embeddings).float())
            self.embedding.weight.requires_grad = True
        self.lstm = nn.LSTM(
            embedding_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        embedded   = self.embedding(input_ids)
        out, _     = self.lstm(embedded)
        mean_hidden = torch.mean(out, dim=1)
        return self.fc(mean_hidden)

    def forward_from_embeddings(self, embeddings):
        out, _ = self.lstm(embeddings)
        mean_hidden = torch.mean(out, dim=1)
        return self.fc(mean_hidden)
    
def text_to_ids(text: str, vocab: dict, max_length: int) -> list[int]:
    tokens    = tokenize_with_stanza(text)
    token_ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens]
    if len(token_ids) > max_length:
        token_ids = token_ids[:max_length]
    padded = [vocab[PAD_TOKEN]] * (max_length - len(token_ids)) + token_ids
    return padded

def texts_to_tensor(texts: list[str], vocab: dict, max_length: int = MAX_LENGTH) -> torch.Tensor:
    return torch.tensor([text_to_ids(t, vocab, max_length) for t in texts], dtype=torch.long)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length=MAX_LENGTH):
        self.labels = labels
        self.data   = []
        for text in tqdm(texts, desc="Preparing dataset tokens"):
            self.data.append(torch.tensor(text_to_ids(text, vocab, max_length), dtype=torch.long))
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.data[idx],
            "labels"   : torch.tensor(self.labels[idx], dtype=torch.long),
        }

model = LSTMClassifier(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, embedding_matrix, num_classes=2).to(device)

history = {
    "train_loss"    : [],
    "train_acc"     : [],
    "train_prec"    : [],
    "train_rec"     : [],
    "train_f1"      : [],
    "val_loss"      : [],
    "val_acc"       : [],
    "val_prec"      : [],
    "val_rec"       : [],
    "val_f1"        : [],
}

if os.path.exists(LSTM_MODEL_PATH):
    print("Pre-trained LSTM model found - loading weights, skipping training.")
    model.load_state_dict(torch.load(LSTM_MODEL_PATH, map_location=device))
    model.eval()
else:
    train_dataset = ReviewDataset(X_train, y_train, vocab_dict)
    val_dataset   = ReviewDataset(X_val,   y_val,   vocab_dict)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print(f"\nTraining for {NUM_EPOCHS} epochs (lr={LEARNING_RATE})...")
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        tr_preds, tr_labels_all = [], []

        for batch in train_loader:
            optimizer.zero_grad()
            input_ids    = batch["input_ids"].to(device)
            batch_labels = batch["labels"].to(device)
            logits       = model(input_ids)
            loss         = criterion(logits, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            tr_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            tr_labels_all.extend(batch_labels.cpu().numpy())
        avg_train_loss = total_loss / len(train_loader)
        model.eval()
        val_loss_total = 0.0
        vl_preds, vl_labels_all = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids    = batch["input_ids"].to(device)
                batch_labels = batch["labels"].to(device)
                logits       = model(input_ids)
                loss         = criterion(logits, batch_labels)
                val_loss_total += loss.item()
                vl_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())
                vl_labels_all.extend(batch_labels.cpu().numpy())
        avg_val_loss = val_loss_total / len(val_loader)
        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(accuracy_score(tr_labels_all, tr_preds))
        history["train_prec"].append(precision_score(tr_labels_all, tr_preds, average="weighted", zero_division=0))
        history["train_rec"].append(recall_score(tr_labels_all, tr_preds, average="weighted", zero_division=0))
        history["train_f1"].append(f1_score(tr_labels_all, tr_preds, average="weighted", zero_division=0))
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(accuracy_score(vl_labels_all, vl_preds))
        history["val_prec"].append(precision_score(vl_labels_all, vl_preds, average="weighted", zero_division=0))
        history["val_rec"].append(recall_score(vl_labels_all, vl_preds, average="weighted", zero_division=0))
        history["val_f1"].append(f1_score(vl_labels_all, vl_preds, average="weighted", zero_division=0))

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f}  Acc: {history['train_acc'][-1]:.4f}  "
            f"F1: {history['train_f1'][-1]:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}  Acc: {history['val_acc'][-1]:.4f}  "
            f"F1: {history['val_f1'][-1]:.4f}"
        )

    torch.save(model.state_dict(), LSTM_MODEL_PATH)
    print(f"\nModel saved to: {LSTM_MODEL_PATH}")

In [ ]:
model.eval()
X_test_input = X_test_raw.tolist() if hasattr(X_test_raw, 'tolist') else list(X_test_raw)
y_test_input = y_test_raw.tolist() if hasattr(y_test_raw, 'tolist') else list(y_test_raw)
X_test_tensor  = texts_to_tensor(X_test_input, vocab_dict)
y_test_tensor  = torch.tensor(y_test_input, dtype=torch.long)
test_dataset   = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)
test_loader    = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
y_pred_list, y_probs_list = [], []
with torch.no_grad():
    for batch_input, _ in test_loader:
        logits = model(batch_input.to(device))
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        y_pred_list.extend(np.argmax(probs, axis=-1))
        y_probs_list.extend(probs[:, 1])

y_pred  = np.array(y_pred_list)
y_probs = np.array(y_probs_list)

acc_val       = accuracy_score(y_test_input, y_pred)
f1_val        = f1_score(y_test_input, y_pred, average="weighted")
precision_val = precision_score(y_test_input, y_pred, average="weighted")
recall_val    = recall_score(y_test_input, y_pred, average="weighted")
auc_roc_val   = roc_auc_score(y_test_input, y_probs)
cm = confusion_matrix(y_test_input, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity_val = tn / (tn + fp)
precisions_curve, recalls_curve, _ = precision_recall_curve(y_test_input, y_probs)
auc_pr_val  = average_precision_score(y_test_input, y_probs)
clf_report  = classification_report(y_test_input, y_pred, target_names=["negatif", "positif"], output_dict=True, digits=4)

print("=" * 55)
print("     LSTM MODEL EVALUATION RESULTS")
print("=" * 55)
print(f"  Accuracy            : {acc_val:.4f}")
print(f"  F1 (weighted)       : {f1_val:.4f}")
print(f"  Precision (weighted): {precision_val:.4f}")
print(f"  Recall (weighted)   : {recall_val:.4f}")
print(f"  Specificity         : {specificity_val:.4f}")
print(f"  AUC-ROC             : {auc_roc_val:.4f}")
print(f"  AUC-PR              : {auc_pr_val:.4f}")
print()
print(classification_report(y_test_input, y_pred, target_names=["negatif", "positif"], digits=4))

epochs_range = range(1, NUM_EPOCHS + 1)

def _save_epoch_plot(train_vals, val_vals, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    if train_vals:
        ax.plot(epochs_range, train_vals, marker="o", color="steelblue",  label="Train")
        ax.plot(epochs_range, val_vals,   marker="s", color="darkorange", label="Validation")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(linestyle="--", alpha=0.5)
    plt.tight_layout()
    path = os.path.join(EVAL_DIR, filename)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

if history["train_loss"]:
    _save_epoch_plot(history["train_loss"], history["val_loss"], "Loss", "LSTM Loss - Train vs Validation", "lstm_loss.png")
    _save_epoch_plot(history["train_acc"],  history["val_acc"], "Accuracy", "LSTM Accuracy - Train vs Validation", "lstm_accuracy.png")
    _save_epoch_plot(history["train_prec"], history["val_prec"], "Precision", "LSTM Precision - Train vs Validation", "lstm_precision.png")
    _save_epoch_plot(history["train_rec"],  history["val_rec"], "Recall", "LSTM Recall - Train vs Validation", "lstm_recall.png")
    _save_epoch_plot(history["train_f1"],   history["val_f1"], "F1", "LSTM F1 - Train vs Validation", "lstm_f1.png")
else:
    print("Training history not available (model loaded from disk) - skipping epoch plots.")

fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["negatif", "positif"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix (Test Set)", fontsize=13, fontweight="bold")
fig.tight_layout()
cm_path = os.path.join(EVAL_DIR, "lstm_confusion_matrix.png")
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {cm_path}")

fpr, tpr, _ = roc_curve(y_test_input, y_probs)

fig3, (ax3, ax4) = plt.subplots(1, 2, figsize=(12, 5))
fig3.suptitle("ROC Curve & Precision-Recall Curve - LSTM", fontsize=13, fontweight="bold")

ax3.plot(fpr, tpr, color="#7ea5f8", linewidth=2, label=f"AUC-ROC = {auc_roc_val:.4f}")
ax3.plot([0, 1], [0, 1], "k--", linewidth=1)
ax3.set_xlabel("False Positive Rate")
ax3.set_ylabel("True Positive Rate")
ax3.set_title("ROC Curve")
ax3.legend(loc="lower right")
ax3.grid(True, alpha=0.3)

ax4.plot(recalls_curve, precisions_curve, color="#73e39c", linewidth=2, label=f"AUC-PR = {auc_pr_val:.4f}")
ax4.set_xlabel("Recall")
ax4.set_ylabel("Precision")
ax4.set_title("Precision-Recall Curve")
ax4.legend(loc="lower left")
ax4.grid(True, alpha=0.3)

fig3.tight_layout()
roc_pr_path = os.path.join(EVAL_DIR, "lstm_roc_pr_curves.png")
fig3.savefig(roc_pr_path, dpi=150, bbox_inches="tight")
plt.close(fig3)
print(f"Saved: {roc_pr_path}")

all_metrics = {
    "per_epoch": {
        "train_loss"     : history["train_loss"],
        "train_accuracy" : history["train_acc"],
        "train_precision": history["train_prec"],
        "train_recall"   : history["train_rec"],
        "train_f1"       : history["train_f1"],
        "val_loss"       : history["val_loss"],
        "val_accuracy"   : history["val_acc"],
        "val_precision"  : history["val_prec"],
        "val_recall"     : history["val_rec"],
        "val_f1"         : history["val_f1"],
    },
    "test_set": {
        "accuracy"             : round(acc_val,       4),
        "f1_weighted"          : round(f1_val,        4),
        "precision_weighted"   : round(precision_val, 4),
        "recall_weighted"      : round(recall_val,    4),
        "specificity"          : round(specificity_val, 4),
        "auc_roc"              : round(auc_roc_val,   4),
        "auc_pr"               : round(auc_pr_val,    4),
        "confusion_matrix"     : cm.tolist(),
        "classification_report": clf_report,
    },
}
metrics_path = os.path.join(EVAL_DIR, "lstm_all_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)
print(f"Saved: {metrics_path}")

## Phase 3 - Integrated Gradients (IG) Lexicon

In [ ]:
def forward_func_for_captum(embeddings: torch.Tensor) -> torch.Tensor:
    return model.forward_from_embeddings(embeddings)

FILTER_OUT = STOPWORDS | NEGATION_WORDS | set(_string.punctuation) | {PAD_TOKEN, UNK_TOKEN}

def run_integrated_gradients_lstm(reviews, top_k_per_review: int = 3, min_freq: int = 2) -> set:
    ig_local = IntegratedGradients(forward_func_for_captum)
    token_counter = Counter()
    total = 0

    model.eval()
    print(f"Running Integrated Gradients on {len(reviews)} reviews...")

    for i, review in enumerate(reviews):
        if not review or not review.strip():
            continue
        try:
            input_tensor = texts_to_tensor([review], vocab_dict).to(device)
            input_emb    = model.embedding(input_tensor)
            baseline     = torch.zeros_like(input_emb)
            with torch.backends.cudnn.flags(enabled=False):
                attrs, _ = ig_local.attribute(
                    input_emb,
                    baselines=baseline,
                    target=1,
                    n_steps=50,
                    return_convergence_delta=True,
                )

            scores   = attrs.squeeze(0).norm(dim=-1).detach().cpu().numpy()
            ids_list = input_tensor[0].tolist()
            scored = []
            for idx, sc in zip(ids_list, scores):
                word = inv_vocab.get(idx, UNK_TOKEN)
                t = word.lower().strip()
                if t in FILTER_OUT or len(t) < 3 or t.isdigit():
                    continue
                scored.append((t, float(sc)))
            scored.sort(key=lambda x: x[1], reverse=True)
            for t, _ in scored[:top_k_per_review]:
                token_counter[t] += 1
            total += 1
        except Exception:
            continue

        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{len(reviews)}...")

    print(f"Done. {total} reviews processed.")
    candidates = [t for t, c in token_counter.most_common() if c >= min_freq]
    print()
    print(f"Top IG opinion tokens (min_freq={min_freq}):")
    print("-" * 45)
    for t, c in token_counter.most_common(50):
        if c >= min_freq:
            print(f"  {t:<25} freq={c}")

    return set(candidates)

inv_vocab = {v: k for k, v in vocab_dict.items()}

LSTM_LEXICON = run_integrated_gradients_lstm(df_train_pool["review_normalized"].tolist(), top_k_per_review=3, min_freq=2)
print(f"\nLSTM IG lexicon contains {len(LSTM_LEXICON)} tokens.")

lexicon_path = os.path.join(RESULTS_DIR, "lstm_lexicon.json")
with open(lexicon_path, "w", encoding="utf-8") as f:
    json.dump(sorted(LSTM_LEXICON), f, indent=2, ensure_ascii=False)
print(f"Lexicon saved to: {lexicon_path}")

## Phase 4 - Parsing

#### Clause Splitting on Conjunctions and Punctuation

In [ ]:
def split_into_clauses(text: str) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    conj_pattern   = r"\b(?:" + "|".join(map(re.escape, SPLIT_CONJUNCTIONS)) + r")\b"
    processed_text = re.sub(conj_pattern, "|", text, flags=re.IGNORECASE)
    processed_text = re.sub(r"[.,!?;\s*]{2,}|[.,!?;]", "|", processed_text)
    raw_clauses    = processed_text.split("|")
    all_clauses    = []
    for clause in raw_clauses:
        cleaned = re.sub(r"\s+", " ", clause).strip()
        if cleaned:
            all_clauses.append(cleaned)
    return all_clauses

sample_input  = "pengiriman cepat tapi produk jelek,, packing rapi dan admin ramah .. ."
sample_output = split_into_clauses(sample_input)
print(f"Clause splitting example")
print(f"INPUT  : {sample_input}")
print(f"OUTPUT : {sample_output}")

#### POS & Dependency Parsing + Phrase Extraction (Stanza)

In [ ]:
def pos_and_dep_parse(text: str) -> list[dict]:
    if not isinstance(text, str) or not text.strip():
        return []
    doc    = nlp_stanza(text)
    tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            tokens.append({
                "id"     : word.id - 1,
                "word"   : word.lemma.lower() if word.lemma else word.text.lower(),
                "pos"    : word.upos,
                "dep"    : word.deprel,
                "head_id": word.head - 1,
            })
    return tokens

def extract_phrases(clause: str) -> list[dict]:
    tokens = pos_and_dep_parse(clause)
    if not tokens:
        return []

    phrases  = []
    seen     = set()
    n        = len(tokens)
    NOUN_POS = {"NOUN", "PROPN"}
    ADJ_POS  = {"ADJ"}
    ADV_POS  = {"ADV"}
    VERB_POS = {"VERB"}

    def is_neg(tok):
        return tok["word"] in NEGATION_WORDS

    def add(tokens_in_phrase, pattern):
        text = " ".join(t["word"] for t in tokens_in_phrase)
        if text not in seen:
            seen.add(text)
            phrases.append({"phrase": text, "pattern": pattern})

    # Rule 1: NOUN + ADJ
    for i in range(n - 1):
        if tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in ADJ_POS:
            if not is_neg(tokens[i]) and not is_neg(tokens[i+1]):
                add([tokens[i], tokens[i+1]], "R1_NOUN+ADJ")
    # Rule 2: NOUN + ADV + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in ADV_POS and tokens[i+2]["pos"] in ADJ_POS and not is_neg(tokens[i+1])):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R2_NOUN+ADV+ADJ")
    # Rule 3: NOUN + NEG + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and is_neg(tokens[i+1]) and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R3_NOUN+NEG+ADJ")
    # Rule 4: NOUN + NEG + ADV + ADJ
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS and is_neg(tokens[i+1]) and tokens[i+2]["pos"] in ADV_POS and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R4_NOUN+NEG+ADV+ADJ")
    # Rule 5: ADJ + NOUN (inversion)
    for i in range(n - 1):
        if tokens[i]["pos"] in ADJ_POS and tokens[i+1]["pos"] in NOUN_POS:
            if tokens[i+1]["dep"] in {"root", "nsubj"}:
                add([tokens[i+1], tokens[i]], "R5_ADJ+NOUN")
    # Rule 6: NOUN + VERB + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in VERB_POS and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R6_NOUN+VERB+ADJ")
    # Rule 7: NOUN + VERB + NEG + ADJ
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in VERB_POS and is_neg(tokens[i+2]) and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R7_NOUN+VERB+NEG+ADJ")
    # Rule 8: NOUN + NOUN + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in NOUN_POS and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R8_NOUN+NOUN+ADJ")
    # Rule 9: NOUN + NOUN + NEG + ADJ
    for i in range(n - 3):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in NOUN_POS and is_neg(tokens[i+2]) and tokens[i+3]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]], "R9_NOUN+NOUN+NEG+ADJ")
    # Rule 10: NOUN + ADJ + ADJ
    for i in range(n - 2):
        if (tokens[i]["pos"] in NOUN_POS and tokens[i+1]["pos"] in ADJ_POS and tokens[i+2]["pos"] in ADJ_POS):
            add([tokens[i], tokens[i+1], tokens[i+2]], "R10_NOUN+ADJ+ADJ")
    # Rule 11: Dependency Conjunction Tree
    for token in tokens:
        if token["dep"] == "conj":
            head_id = token["head_id"]
            if 0 <= head_id < n:
                head = tokens[head_id]
                if head["pos"] in NOUN_POS and token["pos"] in ADJ_POS:
                    add([head, token], "R11_conj+shared_head")
                elif head["pos"] in ADJ_POS and token["pos"] in ADJ_POS:
                    for other in tokens:
                        if other["pos"] in NOUN_POS and other["head_id"] == head_id:
                            add([other, head, token], "R11_conj+shared_head")
                            break

    return phrases

test_clause = "pengiriman sangat cepat"
print("\nPOS + Dependency parse:")
for t in pos_and_dep_parse(test_clause):
    print(f"  {t['word']:15} POS={t['pos']:6} DEP={t['dep']}")
print("\nPhrase extraction:")
for p in extract_phrases(test_clause):
    print(f"  {p}")

## Phase 5 - IG Lexicon Filter

In [ ]:
def filter_by_lstm_lexicon(phrase_list: list[dict]) -> list[dict]:
    if not LSTM_LEXICON:
        print("Warning: LSTM_LEXICON is empty. Skipping filter.")
        return phrase_list
    filtered = []
    for p in phrase_list:
        words_in_phrase = p["phrase"].lower().split()
        if any(w in LSTM_LEXICON for w in words_in_phrase):
            filtered.append(p)
    return filtered

## Phase 6 - Phrase Sentiment Prediction

In [ ]:
def predict_sentiment_lstm_batch(texts: list[str]) -> list[dict]:
    if not texts:
        return []
    model.eval()
    input_ids = texts_to_tensor(texts, vocab_dict).to(device)
    results   = []
    with torch.no_grad():
        probs = torch.softmax(model(input_ids), dim=-1).cpu().numpy()
        for prob in probs:
            label_id = int(np.argmax(prob))
            results.append({"label": ID2LABEL[label_id], "score": float(prob[label_id])})
    return results

def predict_phrases_sentiment(phrase_list: list[dict]) -> list[dict]:
    if not phrase_list:
        return []
    texts       = [p["phrase"] for p in phrase_list]
    predictions = predict_sentiment_lstm_batch(texts)
    return [
        {
            "phrase"   : pd_item["phrase"],
            "sentiment": pred["label"],
            "score"    : pred["score"],
            "pattern"  : pd_item["pattern"],
        }
        for pd_item, pred in zip(phrase_list, predictions)
    ]

## Phase 7 - Semantic Deduplication (FastText cosine similarity)

In [ ]:
def get_phrase_embeddings_ft(phrases: list[str]) -> np.ndarray:
    if not phrases:
        return np.empty((0, EMBEDDING_DIM))
    phrase_vectors = []
    for phrase in phrases:
        words   = phrase.split()
        vectors = []
        for w in words:
            try:
                vectors.append(ft_model[w])
            except KeyError:
                pass
        if vectors:
            phrase_vectors.append(np.mean(vectors, axis=0))
        else:
            phrase_vectors.append(np.zeros(EMBEDDING_DIM))
    return np.array(phrase_vectors)

def semantic_deduplication(phrase_sentiment_list: list[dict], threshold: float = SIMILARITY_THRESHOLD) -> list[dict]:
    if not phrase_sentiment_list:
        return []

    def cluster_phrases(phrases):
        if not phrases:
            return []
        texts          = [p["phrase"] for p in phrases]
        freq           = Counter(texts)
        unique_phrases = list(freq.keys())
        unique_counts  = [freq[p] for p in unique_phrases]
        if len(unique_phrases) <= 1:
            return [{"phrase": unique_phrases[0], "count": unique_counts[0]}] if unique_phrases else []
        embeddings = get_phrase_embeddings_ft(unique_phrases)
        sim_matrix = cosine_similarity(embeddings)
        merged     = [False] * len(unique_phrases)
        result     = []
        for i in range(len(unique_phrases)):
            if merged[i]:
                continue
            cluster_count = unique_counts[i]
            for j in range(i + 1, len(unique_phrases)):
                if not merged[j] and sim_matrix[i][j] >= threshold:
                    cluster_count += unique_counts[j]
                    merged[j]      = True
            result.append({"phrase": unique_phrases[i], "count": cluster_count})
        return result

    print("Running semantic deduplication using FastText similarity weights...")
    pos_clustered = cluster_phrases([p for p in phrase_sentiment_list if p["sentiment"] == "positif"])
    neg_clustered = cluster_phrases([p for p in phrase_sentiment_list if p["sentiment"] == "negatif"])
    return (
        [{**item, "sentiment": "positif"} for item in pos_clustered] +
        [{**item, "sentiment": "negatif"} for item in neg_clustered]
    )

## Phase 8 - Full Pipeline

In [ ]:
def run_full_pipeline(df: pd.DataFrame, top_n: int = TOP_N_SUMMARY, min_count: int = MIN_COUNT):
    all_phrases = []
    total       = len(df)
    start       = time.time()
    print(f"Processing {total} reviews...\n")

    with tqdm(total=total, desc="Phrase Extraction", unit="review") as pbar:
        for i, (_, row) in enumerate(df.iterrows()):
            review  = row["review_normalized"]
            clauses = split_into_clauses(review) or [review]
            for clause in clauses:
                all_phrases.extend(extract_phrases(clause))
            pbar.update(1)
            if i % 10 == 0:
                elapsed = time.time() - start
                rate    = (i + 1) / elapsed if elapsed > 0 else 0
                eta     = (total - i) / rate if rate > 0 else 0
                pbar.set_postfix(phrases=len(all_phrases), rate=f"{rate:.1f} rev/s", ETA=f"{int(eta//60)}m {int(eta%60):02d}s")

    print()
    print(f"Extraction complete in {int((time.time()-start)//60)}m {int((time.time()-start)%60):02d}s")
    print(f"Total phrases extracted: {len(all_phrases)}")

    if not all_phrases:
        print("No phrases extracted.")
        return None, None, {"total_extracted": 0, "total_accepted": 0, "total_rejected": 0}

    all_phrases = filter_by_lstm_lexicon(all_phrases)
    print(f"Phrases after LSTM Lexicon filter: {len(all_phrases)}")

    print("\nPredicting phrase sentiment...")
    accepted = []
    with tqdm(total=len(all_phrases), desc="Sentiment Prediction", unit="phrase") as pbar2:
        for i in range(0, len(all_phrases), 64):
            batch  = all_phrases[i : i + 64]
            result = predict_phrases_sentiment(batch)
            for res in result:
                if res["score"] >= CONFIDENCE_THRESHOLD:
                    accepted.append(res)
            pbar2.update(len(batch))
            pbar2.set_postfix(accepted=len(accepted),
                              rejected=(i + len(batch)) - len(accepted))

    total_accepted = len(accepted)
    total_rejected = len(all_phrases) - total_accepted
    print(f"\nPhrases accepted: {total_accepted}")
    print(f"Phrases rejected: {total_rejected} (confidence < {CONFIDENCE_THRESHOLD})")

    phrase_stats = {
        "total_extracted": len(all_phrases),
        "total_accepted" : total_accepted,
        "total_rejected" : total_rejected,
    }

    deduplicated = semantic_deduplication(accepted)
    pos_raw = sorted([p for p in deduplicated if p["sentiment"] == "positif"], key=lambda x: x["count"], reverse=True)
    neg_raw = sorted([p for p in deduplicated if p["sentiment"] == "negatif"], key=lambda x: x["count"], reverse=True)

    print(f"Unique positive clusters: {len(pos_raw)}")
    print(f"Unique negative clusters: {len(neg_raw)}")

    for p in pos_raw + neg_raw:
        p["pct"] = round(min(p["count"] / total * 100, 100.0), 1)

    pos_phrases = [p for p in pos_raw[:top_n] if p["count"] >= min_count]
    neg_phrases = [p for p in neg_raw[:top_n] if p["count"] >= min_count]

    elapsed_total = time.time() - start
    print(f"\nPipeline complete in {int(elapsed_total//60)}m {int(elapsed_total%60):02d}s")
    return pos_phrases, neg_phrases, phrase_stats

df_sample = df_raw.reset_index(drop=True)
pos_summary, neg_summary, phrase_stats = run_full_pipeline(df_sample, top_n=TOP_N_SUMMARY, min_count=MIN_COUNT)

if pos_summary is not None and neg_summary is not None:
    print("\n" + "=" * 55)
    print("         AUTOMATED REVIEW SUMMARY")
    print("=" * 55)
    print("\nPOSITIVE SUMMARY")
    print("-" * 55)
    for i, p in enumerate(pos_summary, 1):
        bar = "#" * max(1, int(p["pct"] / 2))
        print(f"  {i:2}. {p['phrase']:<25} {bar:<20} {p['pct']}% ({p['count']} occurrences)")
    print("\nNEGATIVE SUMMARY")
    print("-" * 55)
    for i, p in enumerate(neg_summary, 1):
        bar = "#" * max(1, int(p["pct"] / 2))
        print(f"  {i:2}. {p['phrase']:<25} {bar:<20} {p['pct']}% ({p['count']} occurrences)")
    print("\n" + "=" * 55)

## Phase 9 - Save All Outputs

In [ ]:
phrase_summary = {
    "phrase_stats": phrase_stats,
    "positive"    : pos_summary or [],
    "negative"    : neg_summary or [],
}
phrase_summary_path = os.path.join(RESULTS_DIR, "lstm_phrase_summary.json")
with open(phrase_summary_path, "w", encoding="utf-8") as f:
    json.dump(phrase_summary, f, indent=2, ensure_ascii=False)
print(f"Phrase summary saved to: {phrase_summary_path}")

print("\nAll outputs:")
for out_dir, label in [(MODEL_DIR, "model_weights"), (EVAL_DIR, "evaluation_metrics"), (RESULTS_DIR, "results")]:
    print(f"\n  [{label}]")
    for fname in sorted(os.listdir(out_dir)):
        fpath = os.path.join(out_dir, fname)
        if os.path.isfile(fpath):
            size_kb = os.path.getsize(fpath) / 1024
            print(f"    {fname:<50} {size_kb:>8.1f} KB")